# Human Rater Demographics

Aggregates the self-reported demographic survey from the 12 (of 15) human raters
who responded, normalises free-text fields, and exports a summary table to
`turing/report/data/human_demographics.json` for use by the supplement 5 build script.

In [ ]:
import json
from pathlib import Path
from collections import Counter

import pandas as pd

In [ ]:
TURING_DIR = Path(".").resolve().parents[0]
CSV_PATH = TURING_DIR / "data" / "background of human graders - Turing test of patient simulation.csv"
OUTPUT_PATH = TURING_DIR / "report" / "data" / "human_demographics.json"

N_TOTAL_RATERS = 15

df = pd.read_csv(CSV_PATH)
print(f"Responses: {len(df)} / {N_TOTAL_RATERS}")
df.head()

## Normalise fields

In [ ]:
# --- Gender ---
gender_map = {
    "Female": "Female",
    "Male": "Male",
    "M": "Male",
}
df["gender_norm"] = df["Gender (however you are comfortable sharing it)"].map(gender_map)
print("Gender:", df["gender_norm"].value_counts().to_dict())

In [ ]:
# --- Age band ---
df["age_band"] = df["Age band"]
print("Age band:", df["age_band"].value_counts().to_dict())

In [ ]:
# --- Race/ethnicity ---
# Normalise to broader categories
def normalise_race(raw: str) -> str:
    raw_lower = raw.strip().lower()
    if any(k in raw_lower for k in ["white", "european", "caucasian"]):
        return "White"
    if any(k in raw_lower for k in ["east asian"]):
        return "Asian"
    if any(k in raw_lower for k in ["south asian"]):
        return "Asian"
    if raw_lower == "asian":
        return "Asian"
    if any(k in raw_lower for k in ["middle east", "iranian"]):
        return "Middle Eastern"
    return raw.strip()

df["race_norm"] = df["Racial/ethnic background (however you feel comfortable sharing it)"].apply(normalise_race)
print("Race/ethnicity:", df["race_norm"].value_counts().to_dict())

In [ ]:
# --- Role/expertise ---
role_map = {
    "Data Science": "Data Science",
    "Clinical": "Clinical",
}
df["role_norm"] = df["Type of role"].map(role_map)
print("Role:", df["role_norm"].value_counts().to_dict())

In [ ]:
# --- Seniority ---
def normalise_seniority(raw: str) -> str:
    raw_lower = raw.strip().lower()
    # Map to junior / mid / senior
    if any(k in raw_lower for k in ["intern", "coop", "new grad", "new graduate"]):
        return "Junior (0-2 years)"
    # Try to parse numeric years
    import re
    nums = re.findall(r"(\d+)", raw_lower)
    if nums:
        years = max(int(n) for n in nums)
        if years <= 2:
            return "Junior (0-2 years)"
        elif years <= 7:
            return "Mid (3-7 years)"
        else:
            return "Senior (8+ years)"
    if "mid" in raw_lower:
        return "Mid (3-7 years)"
    return raw.strip()

df["seniority_norm"] = df["Level of seniority (in years of career/experience)"].apply(normalise_seniority)
print("Seniority:", df["seniority_norm"].value_counts().to_dict())

## Build summary table

In [ ]:
def build_category(series: pd.Series, total: int) -> list[dict]:
    """Build sorted list of {value, n, pct} dicts."""
    counts = series.value_counts()
    rows = []
    for val, n in counts.items():
        rows.append({"value": val, "n": int(n), "pct": round(n / total * 100, 1)})
    return rows

n_respondents = len(df)

demographics = {
    "n_total_raters": N_TOTAL_RATERS,
    "n_respondents": n_respondents,
    "categories": [
        {"label": "Gender", "values": build_category(df["gender_norm"], n_respondents)},
        {"label": "Age band", "values": build_category(df["age_band"], n_respondents)},
        {"label": "Race/ethnicity", "values": build_category(df["race_norm"], n_respondents)},
        {"label": "Expertise", "values": build_category(df["role_norm"], n_respondents)},
        {"label": "Seniority", "values": build_category(df["seniority_norm"], n_respondents)},
        {"label": "Native English speaker", "values": build_category(
            df["Are you a native English speaker?"], n_respondents)},
        {"label": "Location", "values": build_category(
            df["Where are you based?"], n_respondents)},
    ],
}

demographics

In [ ]:
# Display as a flat table
rows = []
for cat in demographics["categories"]:
    for v in cat["values"]:
        rows.append({"Category": cat["label"], "Value": v["value"], "n": v["n"], "%": v["pct"]})

summary_df = pd.DataFrame(rows)
print(f"\nHuman rater demographics ({n_respondents}/{N_TOTAL_RATERS} responded)\n")
summary_df

In [ ]:
# Export
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
with open(OUTPUT_PATH, "w") as f:
    json.dump(demographics, f, indent=2)
print(f"Exported to {OUTPUT_PATH}")